Matching SlimAttention form and shortening from v1/test notebook

In [ ]:
# Proof of concept for MatShrink paper
# Usage: python3 matShrink_paper.py

%pip install --quiet git+https://github.com/siddhshah/transformer-tricks.git@perplexity-device-arg # branch containing device and dtype arg updates
import transformer_tricks as tt
import numpy as np
import torch
from transformers import AutoConfig

# defs
def matshrink_vo(param, config, storage_dtype=torch.float16):
  """Apply V-O MatShrink to all layers of a Llama-family model.
     Math in fp64; weights written back in storage_dtype.
     Per KV group, pick M as first dk rows of the first query head's O slice."""
  h, h_kv = config.num_attention_heads, config.num_key_value_heads
  g, dk = h // h_kv, config.head_dim

  for layer in range(config.num_hidden_layers):
    # note: all weights are transposed in tensorfile
    Wv = param[tt.weight('V', layer)].to(torch.float64).numpy().T
    Wo = param[tt.weight('O', layer)].to(torch.float64).numpy().T

    for kv in range(h_kv):
      q0 = kv * g
      M = Wo[q0*dk:(q0+1)*dk, :dk].copy()
      M_inv = np.linalg.inv(M)
      Wv[:, kv*dk:(kv+1)*dk] = Wv[:, kv*dk:(kv+1)*dk] @ M
      for q in range(q0, q0 + g):
        Wo[q*dk:(q+1)*dk, :] = M_inv @ Wo[q*dk:(q+1)*dk, :]

    param[tt.weight('V', layer)] = torch.from_numpy(Wv.T).to(storage_dtype).contiguous()
    param[tt.weight('O', layer)] = torch.from_numpy(Wo.T).to(storage_dtype).contiguous()

def matshrink_repo(repo, out_dir, storage_dtype=torch.float16):
  param = tt.get_param(repo)
  config = AutoConfig.from_pretrained(repo)
  matshrink_vo(param, config, storage_dtype=storage_dtype)
  tt.save_repo(repo, param, config, out_dir)

# setup
tt.quiet_hf()
repo = 'HuggingFaceTB/SmolLM2-135M'

In [ ]:
# check that V@O composition is preserved per head after MatShrink
param_orig = tt.get_param(repo)
param_new  = tt.get_param(repo)
config = AutoConfig.from_pretrained(repo)
matshrink_vo(param_new, config, storage_dtype=torch.float64)

h, h_kv = config.num_attention_heads, config.num_key_value_heads
g, dk = h // h_kv, config.head_dim

for layer in range(config.num_hidden_layers):
  Wv0 = param_orig[tt.weight('V', layer)].to(torch.float64).numpy().T
  Wo0 = param_orig[tt.weight('O', layer)].to(torch.float64).numpy().T
  Wv1 = param_new [tt.weight('V', layer)].to(torch.float64).numpy().T
  Wo1 = param_new [tt.weight('O', layer)].to(torch.float64).numpy().T
  ok = all(np.allclose(Wv0[:, (q//g)*dk:(q//g+1)*dk] @ Wo0[q*dk:(q+1)*dk, :],
                       Wv1[:, (q//g)*dk:(q//g+1)*dk] @ Wo1[q*dk:(q+1)*dk, :])
           for q in range(h))
  print(layer, ':', ok)

In [ ]:
# perplexity: baseline vs MatShrink at three storage precisions
speedup = 16 # you should set this to 1 for full wikitext-2 (still faster on GPU)

for name, dt in [('bf16', torch.bfloat16), ('fp16', torch.float16), ('fp32', torch.float32)]:
  print(f'--- {name} ---')
  print('baseline:  ', end='')
  tt.perplexity(repo, speedup=speedup, device='cuda', dtype=dt)
  out_dir = f'./SmolLM2-135M_matshrink_{name}'
  matshrink_repo(repo, out_dir, storage_dtype=dt)
  print('matshrink: ', end='')
  tt.perplexity(out_dir, speedup=speedup, device='cuda', dtype=dt)